# Entrenamiento v3 — XGBoost + Grid Search + k-fold CV

En este notebook entrenamos un modelo XGBoost con optimización de hiperparámetros
mediante Grid Search y validación cruzada estratificada (k-fold).

**Experimentos anteriores:**
- Exp 0 (notebook 3): LogReg + TF-IDF → F1-macro val: 0.87
- Exp 1 (notebook 5): LogReg + TF-IDF + features manuales → pendiente

**Este notebook (Exp 2):**
1. TF-IDF + features manuales (misma config)
2. Grid Search con StratifiedKFold (k=5) sobre XGBoost
3. Entrenamiento con mejores hiperparámetros
4. Evaluación en validación
5. Guardar artefactos
6. Registro en MLflow (Experimento 2)

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"
functions._DATASET_TAGS = {"dataset_type": "artificial", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [1]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

Train: 210 muestras
Validation: 45 muestras
Test: 45 muestras


## 2. TF-IDF + Features manuales

In [2]:
from functions import crear_tfidf, crear_features_manuales, combinar_features

# TF-IDF (misma config que experimentos anteriores)
tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train, X_val, X_test,
    max_features=5000,
    ngram_range=(1, 2),
)

# Features manuales
feat_train = crear_features_manuales(X_train)
feat_val = crear_features_manuales(X_val)
feat_test = crear_features_manuales(X_test)

# Concatenar
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined = combinar_features(X_val_tfidf, feat_val)
X_test_combined = combinar_features(X_test_tfidf, feat_test)

print(f"\nFeatures totales: {X_train_combined.shape[1]} (TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]})")

Vocabulario TF-IDF: 3773 términos
Forma train: (210, 3773)
Forma validation: (45, 3773)
Forma test: (45, 3773)

Features totales: 3779 (TF-IDF: 3773 + manuales: 6)


## 3. Grid Search con k-fold CV

In [3]:
from functions import grid_search_cv

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
}

best_model, best_params, cv_results, le = grid_search_cv(
    X_train_combined, y_train,
    param_grid=param_grid,
    cv=5,
)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

=== Resultados Grid Search (5-fold CV) ===
Mejor F1-macro CV: 0.8744
Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}


In [4]:
# Top 10 combinaciones del Grid Search
print("Top 10 combinaciones de hiperparámetros:\n")
cols = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
print(cv_results[cols].head(10).to_string(index=False))

Top 10 combinaciones de hiperparámetros:

 rank_test_score  mean_test_score  std_test_score                                                                         params
               1         0.874402        0.043249  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
               2         0.869767        0.045466  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
               3         0.868259        0.031856  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
               4         0.863806        0.033817  {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
               5         0.860228        0.048764  {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
               5         0.860228        0.048764  {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
               7         0.858741        0.051950  {'le

## 4. Evaluación en validación con el mejor modelo

In [5]:
from sklearn.metrics import classification_report, f1_score

y_val_pred_enc = best_model.predict(X_val_combined)
y_val_pred = le.inverse_transform(y_val_pred_enc)

print("=== Resultados en VALIDACIÓN (XGBoost — mejor modelo) ===\n")
print(classification_report(y_val, y_val_pred))

f1_val = f1_score(y_val, y_val_pred, average="macro")
acc_val = (y_val_pred == y_val.values).mean()

print(f"F1-score macro (validación): {f1_val:.4f}")
print(f"Accuracy (validación): {acc_val:.4f}")

=== Resultados en VALIDACIÓN (XGBoost — mejor modelo) ===

                 precision    recall  f1-score   support

    alto_riesgo       1.00      0.93      0.96        14
    inaceptable       0.90      0.82      0.86        11
riesgo_limitado       1.00      0.80      0.89        10
  riesgo_minimo       0.64      0.90      0.75        10

       accuracy                           0.87        45
      macro avg       0.89      0.86      0.86        45
   weighted avg       0.90      0.87      0.87        45

F1-score macro (validación): 0.8647
Accuracy (validación): 0.8667


## 5. Comparativa rápida con experimentos anteriores

In [6]:
from functions import entrenar_modelo_baseline

# Exp 0: baseline
BASELINE_F1 = 0.8698

# Exp 1: LogReg + TF-IDF + features manuales (se calcula aquí directamente)
modelo_v2 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)
y_val_pred_v2 = modelo_v2.predict(X_val_combined)
EXP1_F1 = f1_score(y_val, y_val_pred_v2, average="macro")

print("\n=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Experimento':<40} {'F1-macro val':>12}")
print("-" * 55)
print(f"{'Exp 0: LogReg + TF-IDF':<40} {BASELINE_F1:>12.4f}")
print(f"{'Exp 1: LogReg + TF-IDF + feat. manuales':<40} {EXP1_F1:>12.4f}")
print(f"{'Exp 2: XGBoost + Grid Search + k-fold':<40} {f1_val:>12.4f}")

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.85      0.79      0.81        14
    inaceptable       0.82      0.82      0.82        11
riesgo_limitado       1.00      0.80      0.89        10
  riesgo_minimo       0.62      0.80      0.70        10

       accuracy                           0.80        45
      macro avg       0.82      0.80      0.80        45
   weighted avg       0.82      0.80      0.81        45

F1-score macro (validación): 0.8044

=== COMPARATIVA VALIDACIÓN ===
Experimento                              F1-macro val
-------------------------------------------------------
Exp 0: LogReg + TF-IDF                         0.8698
Exp 1: LogReg + TF-IDF + feat. manuales        0.8044
Exp 2: XGBoost + Grid Search + k-fold          0.8647


## 6. Guardar artefactos

In [7]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

# Guardar modelo XGBoost (con LabelEncoder incluido)
path_xgb = os.path.join(output_dir, "modelo_xgboost.joblib")
joblib.dump(best_model, path_xgb)
print(f"Modelo XGBoost guardado en: {path_xgb}")

# Guardar el vectorizador (mismo que los anteriores)
path_tfidf = os.path.join(output_dir, "tfidf_vectorizer.joblib")
joblib.dump(tfidf, path_tfidf)
print(f"Vectorizador guardado en: {path_tfidf}")

# Guardar LabelEncoder por separado
path_le = os.path.join(output_dir, "label_encoder.joblib")
joblib.dump(le, path_le)
print(f"LabelEncoder guardado en: {path_le}")

Modelo XGBoost guardado en: model\modelo_xgboost.joblib
Vectorizador guardado en: model\tfidf_vectorizer.joblib
LabelEncoder guardado en: model\label_encoder.joblib


## 7. Registro en MLflow — Experimento 2

In [8]:
# MLflow (solo falla esta celda si el servidor no está disponible)
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT, _DATASET_TAGS

# Guard: verificar que los artefactos de la celda anterior existen
_missing = [v for v in ("path_tfidf", "path_le", "best_model") if v not in dir()]
if _missing:
    print(f"⚠ Variables no definidas: {_missing}. Ejecuta primero la celda de guardado de artefactos.")
else:
    try:
        configure_mlflow()
        mlflow.set_experiment(MLFLOW_EXPERIMENT)

        with mlflow.start_run(run_name="v3_xgboost_gridsearch", tags=_DATASET_TAGS):
            mlflow.log_param("tfidf_max_features",  5000)
            mlflow.log_param("tfidf_ngram_range",   "(1, 2)")
            mlflow.log_param("tfidf_sublinear_tf",  True)
            mlflow.log_param("modelo",              "XGBClassifier")
            for param_name, param_val in best_params.items():
                mlflow.log_param(f"xgb_{param_name}", param_val)
            mlflow.log_param("features_manuales",   list(feat_train.columns))
            mlflow.log_param("total_features",      X_train_combined.shape[1])
            mlflow.log_param("cv_folds",            5)

            mlflow.log_metric("best_cv_f1_macro",   cv_results.iloc[0]["mean_test_score"])
            mlflow.log_metric("val_f1_macro",        f1_val)
            mlflow.log_metric("val_accuracy",        acc_val)
            mlflow.log_metric("baseline_f1_macro",   BASELINE_F1)

            mlflow.log_artifact(path_tfidf, artifact_path="artifacts")
            mlflow.log_artifact(path_le, artifact_path="artifacts")
            mlflow.xgboost.log_model(best_model, "model")

            print("✓ Exp 2 (XGBoost) registrado en MLflow")
            print(f"  Best CV F1-macro: {cv_results.iloc[0]['mean_test_score']:.4f}")
            print(f"  Val F1-macro: {f1_val:.4f}")
            print(f"  Run ID: {mlflow.active_run().info.run_id}")
    except (mlflow.exceptions.MlflowException, ConnectionError, OSError) as e:
        print(f"⚠ MLflow no disponible ({type(e).__name__}): {e}")
    except Exception:
        raise


Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.244.146.100/mlflow
⚠ MLflow no disponible (MlflowException): API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.244.146.100'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
